# 00 下载并标准化真实 GRN prior

这个 notebook 只为 **真实数据 real discovery** 准备 prior。

输出文件：
- `resources/grn_prior/collectri_human.tsv`
- `resources/grn_prior/trrust_human.tsv`
- `resources/ccc_prior/ligand_receptor_human.tsv`（由 OmniPath / CellPhoneDB / CellChatDB 等真实 LR 数据库合并）

策略：
- 优先保留已经存在且通过规模/列名校验的真实 prior
- 若缺失则重建：CollecTRI 走 decoupler；TRRUST 优先官方下载，失败时再从完整 CollecTRI 中提取带 TRRUST 来源标记的子集
- 若检测到占位/demo prior（例如只有几行或 `demo_ref`），会直接判为无效并重建


In [ ]:
from pathlib import Path
import importlib
import subprocess
import sys
import io
import gzip
import time
import re
import requests
import pandas as pd

ROOT = Path('..').resolve()
PRIOR_ROOT = ROOT / 'resources' / 'grn_prior'
PRIOR_ROOT.mkdir(parents=True, exist_ok=True)

OVERWRITE_EXISTING = False
REQUEST_TIMEOUT = 30
COLLECTRI_MIN_ROWS = 1000
TRRUST_MIN_ROWS = 100

TRRUST_URLS = [
    'https://www.grnpedia.org/trrust/data/trrust_rawdata.human.tsv',
    'https://www.grnpedia.org/trrust/data/trrust_rawdata.human.tsv.gz',
]

COLLECTRI_REQUIRED = ['source_genesymbol', 'target_genesymbol', 'consensus_stimulation', 'consensus_inhibition', 'references', 'sources']
TRRUST_REQUIRED = ['source_genesymbol', 'target_genesymbol', 'references', 'sources']


def ensure_package(package_name: str, import_name: str | None = None):
    import_name = import_name or package_name
    try:
        return importlib.import_module(import_name)
    except ImportError:
        print(f'[info] installing {package_name} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package_name])
        return importlib.import_module(import_name)


def _read_tsv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep='	', low_memory=False)


def _looks_like_demo(df: pd.DataFrame) -> bool:
    if df.empty:
        return True
    lowered = set(map(str.lower, df.columns.astype(str)))
    if 'references' in lowered:
        ref_col = next(c for c in df.columns if str(c).lower() == 'references')
        refs = df[ref_col].astype(str)
        if refs.str.contains('demo_ref', case=False, na=False).all():
            return True
    return False


def _validate_prior(path: Path, required_cols: list[str], min_rows: int) -> dict:
    df = _read_tsv(path)
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f'{path.name} is missing required columns: {missing}')
    if len(df) < min_rows:
        raise ValueError(f'{path.name} has only {len(df)} rows; expected at least {min_rows}.')
    if _looks_like_demo(df):
        raise ValueError(f'{path.name} looks like a demo/toy prior and should not be used for real discovery.')
    return {'file': str(path), 'n_rows': int(len(df)), 'n_cols': int(df.shape[1])}


def _normalize_collectri(df: pd.DataFrame) -> pd.DataFrame:
    rename = {
        'source': 'source_genesymbol',
        'target': 'target_genesymbol',
        'tf': 'source_genesymbol',
        'target_gene': 'target_genesymbol',
        'genesymbol': 'source_genesymbol',
        'mor': 'consensus_stimulation',
        'sign_decision': 'consensus_stimulation',
        'resource': 'sources',
        'resources': 'sources',
        'pmid': 'references',
        'reference': 'references',
    }
    out = df.rename(columns={k: v for k, v in rename.items() if k in df.columns}).copy()
    for col in ['source_genesymbol', 'target_genesymbol']:
        if col not in out.columns:
            raise ValueError(f'CollecTRI table missing column: {col}')
    stim_raw = None
    if 'consensus_stimulation' in out.columns:
        stim_raw = out['consensus_stimulation']
    elif 'weight' in out.columns:
        stim_raw = out['weight']
    elif 'mode' in out.columns:
        stim_raw = out['mode']
    if stim_raw is None:
        out['consensus_stimulation'] = 0
        out['consensus_inhibition'] = 0
    else:
        s = stim_raw.astype(str).str.lower()
        pos = s.isin(['1', '1.0', 'true', 't', 'activate', 'activation', 'stimulation', 'up', 'positive'])
        neg = s.isin(['-1', '-1.0', 'false', 'f', 'repress', 'repression', 'inhibition', 'down', 'negative'])
        num = pd.to_numeric(stim_raw, errors='coerce')
        pos = pos | (num > 0).fillna(False)
        neg = neg | (num < 0).fillna(False)
        out['consensus_stimulation'] = pos.astype(int)
        out['consensus_inhibition'] = neg.astype(int)
    if 'references' not in out.columns:
        out['references'] = 'NA'
    out['references'] = out['references'].fillna('NA').astype(str)
    if 'sources' not in out.columns:
        out['sources'] = 'CollecTRI'
    out['sources'] = out['sources'].fillna('CollecTRI').astype(str)
    out = out[COLLECTRI_REQUIRED].copy()
    out['source_genesymbol'] = out['source_genesymbol'].astype(str)
    out['target_genesymbol'] = out['target_genesymbol'].astype(str)
    out = out.dropna(subset=['source_genesymbol', 'target_genesymbol']).drop_duplicates().reset_index(drop=True)
    return out


def _normalize_trrust(df: pd.DataFrame) -> pd.DataFrame:
    rename = {
        'source': 'source_genesymbol',
        'target': 'target_genesymbol',
        'tf': 'source_genesymbol',
        'target_gene': 'target_genesymbol',
        'pmid': 'references',
        'reference': 'references',
        'resource': 'sources',
        'resources': 'sources',
        'source_genesymbol ': 'source_genesymbol',
    }
    out = df.rename(columns={k: v for k, v in rename.items() if k in df.columns}).copy()

    if {'TF', 'Target', 'Mode'}.issubset(out.columns):
        out = out.rename(columns={'TF': 'source_genesymbol', 'Target': 'target_genesymbol'})
        if 'PMID' in out.columns:
            out = out.rename(columns={'PMID': 'references'})
        else:
            out['references'] = 'TRRUST'
        out['sources'] = 'TRRUST'
    
    for col in ['source_genesymbol', 'target_genesymbol']:
        if col not in out.columns:
            raise ValueError(f'TRRUST table missing column: {col}')
    if 'references' not in out.columns:
        out['references'] = 'TRRUST'
    if 'sources' not in out.columns:
        out['sources'] = 'TRRUST'
    out = out[TRRUST_REQUIRED].copy()
    out['source_genesymbol'] = out['source_genesymbol'].astype(str)
    out['target_genesymbol'] = out['target_genesymbol'].astype(str)
    out['references'] = out['references'].fillna('TRRUST').astype(str)
    out['sources'] = out['sources'].fillna('TRRUST').astype(str)
    out = out.dropna(subset=['source_genesymbol', 'target_genesymbol']).drop_duplicates().reset_index(drop=True)
    return out


def _download_bytes(url: str) -> bytes:
    headers = {'User-Agent': 'Mozilla/5.0'}
    r = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    raw = r.content
    head = raw[:1024].lower()
    if any(tag in head for tag in [b'<html', b'<!doctype html', b'<head', b'<body']):
        raise ValueError(f'{url} returned HTML, not the raw table.')
    if url.endswith('.gz') or raw[:2] == bytes([0x1f, 0x8b]):
        raw = gzip.decompress(raw)
    return raw


def export_collectri(out_path: Path, overwrite: bool = OVERWRITE_EXISTING):
    if out_path.exists() and not overwrite:
        try:
            info = _validate_prior(out_path, COLLECTRI_REQUIRED, COLLECTRI_MIN_ROWS)
            info['source'] = 'existing_valid_file'
            return info
        except Exception as e:
            print(f'[warn] existing {out_path.name} rejected -> {e}')

    decoupler = ensure_package('decoupler')
    df = None
    last_error = None
    candidate_calls = []
    if hasattr(decoupler, 'op') and hasattr(decoupler.op, 'collectri'):
        candidate_calls.append(lambda: decoupler.op.collectri(organism='human'))
        candidate_calls.append(lambda: decoupler.op.collectri(organism='human', license='academic'))
    if hasattr(decoupler, 'get_collectri'):
        candidate_calls.append(lambda: decoupler.get_collectri(organism='human'))
    if hasattr(decoupler, 'get_progeny'):
        pass
    for fn in candidate_calls:
        try:
            df = fn()
            break
        except Exception as e:
            last_error = e
            print(f'[warn] CollecTRI fetch attempt failed -> {e}')
    if df is None:
        raise RuntimeError(f'Unable to fetch CollecTRI via decoupler. Last error: {last_error}')
    df = _normalize_collectri(df)
    if len(df) < COLLECTRI_MIN_ROWS:
        raise RuntimeError(f'Fetched CollecTRI is unexpectedly small ({len(df)} rows).')
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, sep='	', index=False)
    info = _validate_prior(out_path, COLLECTRI_REQUIRED, COLLECTRI_MIN_ROWS)
    info['source'] = 'decoupler_collectri'
    return info


def export_trrust(out_path: Path, collectri_path: Path, overwrite: bool = OVERWRITE_EXISTING):
    if out_path.exists() and not overwrite:
        try:
            info = _validate_prior(out_path, TRRUST_REQUIRED, TRRUST_MIN_ROWS)
            info['source'] = 'existing_valid_file'
            return info
        except Exception as e:
            print(f'[warn] existing {out_path.name} rejected -> {e}')

    last_error = None
    for url in TRRUST_URLS:
        try:
            raw = _download_bytes(url)
            text = raw.decode('utf-8', errors='replace')
            df = pd.read_csv(io.StringIO(text), sep='	', low_memory=False)
            df = _normalize_trrust(df)
            if len(df) < TRRUST_MIN_ROWS:
                raise ValueError(f'official TRRUST download too small: {len(df)} rows')
            df.to_csv(out_path, sep='	', index=False)
            info = _validate_prior(out_path, TRRUST_REQUIRED, TRRUST_MIN_ROWS)
            info['source'] = 'official_trrust_download'
            return info
        except Exception as e:
            last_error = e
            print(f'[warn] failed TRRUST URL: {url} -> {e}')
            time.sleep(1)

    if not collectri_path.exists():
        raise RuntimeError(f'TRRUST download failed and collectri file is unavailable for fallback. Last error: {last_error}')

    collectri = _read_tsv(collectri_path)
    src_col = 'sources' if 'sources' in collectri.columns else None
    if src_col is None:
        raise RuntimeError('TRRUST download failed and collectri fallback lacks a sources column.')
    mask = collectri[src_col].astype(str).str.contains(r'(^|[;|, ])TRRUST([;|, ]|$)', case=False, na=False)
    df = collectri.loc[mask, ['source_genesymbol', 'target_genesymbol', 'references', 'sources']].drop_duplicates().reset_index(drop=True)
    if len(df) < TRRUST_MIN_ROWS:
        raise RuntimeError(f'TRRUST download failed and collectri-derived TRRUST subset is too small ({len(df)} rows). Last error: {last_error}')
    df['sources'] = 'TRRUST_from_CollecTRI'
    df.to_csv(out_path, sep='	', index=False)
    info = _validate_prior(out_path, TRRUST_REQUIRED, TRRUST_MIN_ROWS)
    info['source'] = 'collectri_trrust_subset_fallback'
    return info


In [ ]:
logs = []

collectri_path = PRIOR_ROOT / 'collectri_human.tsv'
logs.append(export_collectri(collectri_path))
print(f'[ok] collectri_human.tsv ready -> {collectri_path}')

trrust_path = PRIOR_ROOT / 'trrust_human.tsv'
logs.append(export_trrust(trrust_path, collectri_path=collectri_path))
print(f'[ok] trrust_human.tsv ready -> {trrust_path}')

display(pd.DataFrame(logs))
for p in [collectri_path, trrust_path]:
    df = pd.read_csv(p, sep='	')
    print(f'\n{p.name}: shape={df.shape}')
    display(df.head(3))


## 下载并合并真实 ligand–receptor prior（OmniPath / CellPhoneDB / CellChatDB）

这一段用于替换早期的 `resources/ccc_prior/ligand_receptor_seed_crc.tsv` 小型 seed prior。运行后会在 `resources/ccc_prior/` 下生成：

- `omnipath_ligand_receptor_human.tsv`
- `cellphonedb_ligand_receptor_human.tsv`（若 GitHub 数据结构可解析）
- `cellchat_ligand_receptor_human.tsv`（若本机可用 R/Rscript 或 pyreadr）
- `ligand_receptor_human.tsv`：合并后的主 LR prior，pipeline 默认优先读取这个文件
- `ccc_prior_download_log.csv`：下载与解析日志

注意：CellChatDB 是 R 对象，完整解析通常需要本机安装 R；若只成功下载 OmniPath，也已经是正式公开数据库 prior，不再使用 toy seed prior。

In [ ]:
# Build the CCC prior used by ccc_model.py.
# Default behavior: try real databases first; if the local network blocks all real
# sources, fall back to the small CRC seed only so the pipeline can smoke-test.
# For final/formal analysis, the download log below must show OmniPath/CellPhoneDB/CellChatDB rows with n_pairs > 0.

from pathlib import Path
import subprocess, sys
import pandas as pd

ROOT = Path('..').resolve()
outdir = ROOT / 'resources' / 'ccc_prior'
cmd = [
    sys.executable,
    str(ROOT / 'scripts' / 'download_ccc_prior.py'),
    '--outdir', str(outdir),
    '--timeout', '20',
    '--allow-seed-fallback',
]
print(' '.join(cmd))
proc = subprocess.run(cmd, text=True, capture_output=True)
print(proc.stdout)
if proc.stderr:
    print(proc.stderr)
if proc.returncode != 0:
    raise RuntimeError(
        'CCC prior download failed. Check internet/proxy/SSL access to OmniPath/GitHub, '
        'or run scripts/download_ccc_prior.py manually with --allow-seed-fallback for smoke testing.'
    )

ccc_prior = outdir / 'ligand_receptor_human.tsv'
ccc_log = outdir / 'ccc_prior_download_log.csv'
log = pd.read_csv(ccc_log) if ccc_log.exists() else pd.DataFrame()
display(log)

real_ok = False
if not log.empty and {'database', 'n_pairs'}.issubset(log.columns):
    real_ok = bool(log[log['database'].isin(['OmniPath', 'CellPhoneDB', 'CellChatDB'])]['n_pairs'].fillna(0).astype(int).gt(0).any())

if not real_ok:
    print('[warning] 本次没有成功解析真实 LR 数据库，当前 ligand_receptor_human.tsv 来自 seed fallback；只能用于流程测试，不能作为正式分析。')
else:
    print('[ok] 至少一个真实 LR 数据库解析成功，已生成正式 ligand_receptor_human.tsv。')

print(f'CCC prior ready: {ccc_prior}')
print(f'Download log: {ccc_log}')


In [ ]:
# Inspect the merged LR prior before running the real pipeline.
import pandas as pd
from pathlib import Path
ROOT = Path('..').resolve()
ccc_prior = ROOT / 'resources' / 'ccc_prior' / 'ligand_receptor_human.tsv'
ccc_log = ROOT / 'resources' / 'ccc_prior' / 'ccc_prior_download_log.csv'

lr = pd.read_csv(ccc_prior, sep='	')
print('merged LR prior shape:', lr.shape)
print('n ligands:', lr['ligand'].nunique() if 'ligand' in lr else 0)
print('n receptors:', lr['receptor'].nunique() if 'receptor' in lr else 0)
print('databases:', sorted(set(';'.join(lr.get('databases', pd.Series(dtype=str)).dropna().astype(str)).split(';'))))
display(pd.read_csv(ccc_log).head(10))
display(lr.head(10))